Case de estudio 3 - RapidX: Pipeline JSON → Parquet → BI

In [1]:
// CELDA 1 — Inicialización
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File

val spark = SparkSession.builder()
  .appName("RapidX_BI_Pipeline")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()

import spark.implicits._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo — RapidX BI Pipeline")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 13:49:32 INFO SparkContext: Running Spark version 4.1.1
26/05/05 13:49:32 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/05 13:49:32 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/05/05 13:49:32 INFO ResourceUtils: ==============================================================
26/05/05 13:49:32 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/05 13:49:32 INFO ResourceUtils: ==============================================================
26/05/05 13:49:32 INFO SparkContext: Submitted application: RapidX_BI_Pipeline
26/05/05 13:49:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/05 13:49:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/05 13:49:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/05 13:49:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/05 13:49:32 INFO SecurityManager: Changing view

2026-05-05T11:49:32.904152700Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@626ce3a6
import spark.implicits._

In [2]:
// CELDA 2 — Carpetas de trabajo
val rutaBase   = "C:/Curso-Scala/datos/rapidx"
val rutaJSON   = s"$rutaBase/json_raw"
val rutaSalida = s"$rutaBase/salida"

List(rutaJSON, rutaSalida).foreach { c =>
  Files.createDirectories(Paths.get(c))
  println(s"  📁 $c")
}

println("\n✅ Estructura de carpetas lista")

  📁 C:/Curso-Scala/datos/rapidx/json_raw
  📁 C:/Curso-Scala/datos/rapidx/salida

✅ Estructura de carpetas lista


rutaBase: String = "C:/Curso-Scala/datos/rapidx"
rutaJSON: String = "C:/Curso-Scala/datos/rapidx/json_raw"
rutaSalida: String = "C:/Curso-Scala/datos/rapidx/salida"

In [3]:
// CELDA 3.1 — JSON Madrid día 1
val jsonMadrid1 =
"""[
  {"viaje_id":"VJ-M-0001","timestamp_inicio":"2024-11-01T07:12:33","timestamp_fin":"2024-11-01T07:38:55","ciudad":"Madrid","zona_origen":"Chamberí","zona_destino":"Retiro","distancia_km":4.8,"duracion_min":26,"tarifa_base":6.40,"propina":1.20,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0002","timestamp_inicio":"2024-11-01T08:05:10","timestamp_fin":"2024-11-01T08:29:40","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Moncloa","distancia_km":3.2,"duracion_min":24,"tarifa_base":5.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.2,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0003","timestamp_inicio":"2024-11-01T08:45:00","timestamp_fin":null,"ciudad":"Madrid","zona_origen":"Vallecas","zona_destino":"Barajas","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-119","pasajero_id":"PAX-5512","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0004","timestamp_inicio":"2024-11-01T09:30:15","timestamp_fin":"2024-11-01T10:05:50","ciudad":"Madrid","zona_origen":"Salamanca","zona_destino":"Tetuán","distancia_km":6.1,"duracion_min":35,"tarifa_base":9.20,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-7720","tipo_vehiculo":"UberComfort","surge_multiplier":1.5,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0005","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":"2024-11-01T11:18:30","ciudad":"Madrid","zona_origen":"Arganzuela","zona_destino":"Centro","distancia_km":2.9,"duracion_min":18,"tarifa_base":4.80,"propina":0.50,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-332","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0006","timestamp_inicio":"2024-11-01T12:15:44","timestamp_fin":"2024-11-01T12:55:10","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Chamartín","distancia_km":11.3,"duracion_min":39,"tarifa_base":14.50,"propina":3.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0007","timestamp_inicio":"2024-11-01T14:00:00","timestamp_fin":"2024-11-01T14:22:00","ciudad":"Madrid","zona_origen":"Hortaleza","zona_destino":"Retiro","distancia_km":7.7,"duracion_min":22,"tarifa_base":10.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":5,"conductor_id":"DRV-885","pasajero_id":"PAX-2230","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0008","timestamp_inicio":"2024-11-01T17:30:00","timestamp_fin":"2024-11-01T18:10:20","ciudad":"Madrid","zona_origen":"Centro","zona_destino":"Vallecas","distancia_km":8.9,"duracion_min":40,"tarifa_base":12.00,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-4410","tipo_vehiculo":"UberComfort","surge_multiplier":2.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0009","timestamp_inicio":"2024-11-01T19:45:00","timestamp_fin":null,"ciudad":"Madrid","zona_origen":"Moncloa","zona_destino":"Sol","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-332","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0010","timestamp_inicio":"2024-11-01T22:10:05","timestamp_fin":"2024-11-01T22:45:30","ciudad":"Madrid","zona_origen":"Retiro","zona_destino":"Salamanca","distancia_km":3.5,"duracion_min":35,"tarifa_base":7.80,"propina":1.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.8,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"}
]"""

val ruta1 = s"$rutaJSON/viajes_madrid_2024-11-01.json"
Files.write(Paths.get(ruta1), jsonMadrid1.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta1")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_madrid_2024-11-01.json


jsonMadrid1: String = """[
  {"viaje_id":"VJ-M-0001","timestamp_inicio":"2024-11-01T07:12:33","timestamp_fin":"2024-11-01T07:38:55","ciudad":"Madrid","zona_origen":"Chamberí","zona_destino":"Retiro","distancia_km":4.8,"duracion_min":26,"tarifa_base":6.40,"propina":1.20,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0002","timestamp_inicio":"2024-11-01T08:05:10","timestamp_fin":"2024-11-01T08:29:40","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Moncloa","distancia_km":3.2,"duracion_min":24,"tarifa_base":5.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.2,"_sys_version":"v3.2","_batch_id":"BAT-20241

In [4]:
// CELDA 3.2 — JSON Madrid día 2
val jsonMadrid2 =
"""[
  {"viaje_id":"VJ-M-0011","timestamp_inicio":"2024-11-02T06:55:00","timestamp_fin":"2024-11-02T07:25:10","ciudad":"Madrid","zona_origen":"Tetuán","zona_destino":"Chamartín","distancia_km":4.1,"duracion_min":30,"tarifa_base":6.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0012","timestamp_inicio":"2024-11-02T08:20:00","timestamp_fin":"2024-11-02T09:00:45","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Centro","distancia_km":14.2,"duracion_min":40,"tarifa_base":18.50,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-7720","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0013","timestamp_inicio":"2024-11-02T10:00:00","timestamp_fin":"2024-11-02T10:19:30","ciudad":"Madrid","zona_origen":"Retiro","zona_destino":"Arganzuela","distancia_km":3.3,"duracion_min":19,"tarifa_base":5.60,"propina":1.00,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-885","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0014","timestamp_inicio":"2024-11-02T13:30:00","timestamp_fin":"2024-11-02T14:10:00","ciudad":"Madrid","zona_origen":"Salamanca","zona_destino":"Vallecas","distancia_km":9.2,"duracion_min":40,"tarifa_base":13.40,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-5512","tipo_vehiculo":"UberComfort","surge_multiplier":1.3,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0015","timestamp_inicio":"2024-11-02T18:45:00","timestamp_fin":"2024-11-02T19:30:00","ciudad":"Madrid","zona_origen":"Centro","zona_destino":"Moncloa","distancia_km":5.5,"duracion_min":45,"tarifa_base":9.90,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-332","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.9,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"}
]"""

val ruta2 = s"$rutaJSON/viajes_madrid_2024-11-02.json"
Files.write(Paths.get(ruta2), jsonMadrid2.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta2")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_madrid_2024-11-02.json


jsonMadrid2: String = """[
  {"viaje_id":"VJ-M-0011","timestamp_inicio":"2024-11-02T06:55:00","timestamp_fin":"2024-11-02T07:25:10","ciudad":"Madrid","zona_origen":"Tetuán","zona_destino":"Chamartín","distancia_km":4.1,"duracion_min":30,"tarifa_base":6.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0012","timestamp_inicio":"2024-11-02T08:20:00","timestamp_fin":"2024-11-02T09:00:45","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Centro","distancia_km":14.2,"duracion_min":40,"tarifa_base":18.50,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-7720","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT

In [5]:
// CELDA 3.3 — JSON Barcelona
val jsonBarcelona =
"""[
  {"viaje_id":"VJ-B-0001","timestamp_inicio":"2024-11-01T07:30:00","timestamp_fin":"2024-11-01T08:00:15","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Eixample","distancia_km":3.1,"duracion_min":30,"tarifa_base":5.80,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:35:20","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Barceloneta","distancia_km":5.7,"duracion_min":35,"tarifa_base":8.30,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":3,"conductor_id":"DRV-730","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0003","timestamp_inicio":"2024-11-01T10:10:00","timestamp_fin":null,"ciudad":"Barcelona","zona_origen":"Sagrada Familia","zona_destino":"Gràcia","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-610","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0004","timestamp_inicio":"2024-11-01T12:00:00","timestamp_fin":"2024-11-01T12:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Aeropuerto BCN","distancia_km":15.8,"duracion_min":50,"tarifa_base":22.00,"propina":5.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-850","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0005","timestamp_inicio":"2024-11-01T14:30:00","timestamp_fin":"2024-11-01T14:55:10","ciudad":"Barcelona","zona_origen":"Barceloneta","zona_destino":"Poblenou","distancia_km":4.2,"duracion_min":25,"tarifa_base":6.90,"propina":1.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-730","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0006","timestamp_inicio":"2024-11-01T18:00:00","timestamp_fin":"2024-11-01T18:40:00","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Sagrada Familia","distancia_km":6.3,"duracion_min":40,"tarifa_base":10.20,"propina":2.00,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-3341","tipo_vehiculo":"UberComfort","surge_multiplier":1.6,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0007","timestamp_inicio":"2024-11-01T20:15:00","timestamp_fin":"2024-11-01T20:45:30","ciudad":"Barcelona","zona_origen":"Poblenou","zona_destino":"Eixample","distancia_km":5.0,"duracion_min":30,"tarifa_base":8.50,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-850","pasajero_id":"PAX-7720","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"}
]"""

val ruta3 = s"$rutaJSON/viajes_barcelona_2024-11-01.json"
Files.write(Paths.get(ruta3), jsonBarcelona.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta3")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_barcelona_2024-11-01.json


jsonBarcelona: String = """[
  {"viaje_id":"VJ-B-0001","timestamp_inicio":"2024-11-01T07:30:00","timestamp_fin":"2024-11-01T08:00:15","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Eixample","distancia_km":3.1,"duracion_min":30,"tarifa_base":5.80,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:35:20","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Barceloneta","distancia_km":5.7,"duracion_min":35,"tarifa_base":8.30,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":3,"conductor_id":"DRV-730","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_

In [6]:
// CELDA 3.4 — JSON Sevilla
val jsonSevilla =
"""[
  {"viaje_id":"VJ-S-0001","timestamp_inicio":"2024-11-01T08:00:00","timestamp_fin":"2024-11-01T08:22:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Centro","distancia_km":3.5,"duracion_min":22,"tarifa_base":5.20,"propina":0.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0002","timestamp_inicio":"2024-11-01T09:30:00","timestamp_fin":"2024-11-01T10:05:00","ciudad":"Sevilla","zona_origen":"Macarena","zona_destino":"Aeropuerto SVQ","distancia_km":9.8,"duracion_min":35,"tarifa_base":12.50,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-4410","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0003","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":null,"ciudad":"Sevilla","zona_origen":"Nervión","zona_destino":"Triana","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-561","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0004","timestamp_inicio":"2024-11-01T13:15:00","timestamp_fin":"2024-11-01T13:40:00","ciudad":"Sevilla","zona_origen":"Centro","zona_destino":"Nervión","distancia_km":4.2,"duracion_min":25,"tarifa_base":6.30,"propina":1.00,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-623","pasajero_id":"PAX-2230","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0005","timestamp_inicio":"2024-11-01T17:00:00","timestamp_fin":"2024-11-01T17:28:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Macarena","distancia_km":5.1,"duracion_min":28,"tarifa_base":7.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-6631","tipo_vehiculo":"UberComfort","surge_multiplier":1.4,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"}
]"""

val ruta4 = s"$rutaJSON/viajes_sevilla_2024-11-01.json"
Files.write(Paths.get(ruta4), jsonSevilla.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta4")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_sevilla_2024-11-01.json


jsonSevilla: String = """[
  {"viaje_id":"VJ-S-0001","timestamp_inicio":"2024-11-01T08:00:00","timestamp_fin":"2024-11-01T08:22:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Centro","distancia_km":3.5,"duracion_min":22,"tarifa_base":5.20,"propina":0.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0002","timestamp_inicio":"2024-11-01T09:30:00","timestamp_fin":"2024-11-01T10:05:00","ciudad":"Sevilla","zona_origen":"Macarena","zona_destino":"Aeropuerto SVQ","distancia_km":9.8,"duracion_min":35,"tarifa_base":12.50,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-4410","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_

In [7]:
// CELDA 3.6 — JSON Bilbao
val jsonBilbao =
"""[
  {"viaje_id":"VJ-BI-0001","timestamp_inicio":"2024-11-01T07:45:00","timestamp_fin":"2024-11-01T08:10:00","ciudad":"Bilbao","zona_origen":"Casco Viejo","zona_destino":"Guggenheim","distancia_km":2.8,"duracion_min":25,"tarifa_base":5.50,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-910","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:45:00","ciudad":"Bilbao","zona_origen":"Aeropuerto BIO","zona_destino":"Centro","distancia_km":11.5,"duracion_min":45,"tarifa_base":15.80,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0003","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":"2024-11-01T11:22:00","ciudad":"Bilbao","zona_origen":"Guggenheim","zona_destino":"Indautxu","distancia_km":3.3,"duracion_min":22,"tarifa_base":5.90,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-910","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0004","timestamp_inicio":"2024-11-01T14:00:00","timestamp_fin":"2024-11-01T14:35:00","ciudad":"Bilbao","zona_origen":"Indautxu","zona_destino":"Casco Viejo","distancia_km":4.8,"duracion_min":35,"tarifa_base":8.10,"propina":1.50,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-2230","tipo_vehiculo":"UberComfort","surge_multiplier":1.3,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0005","timestamp_inicio":"2024-11-01T17:30:00","timestamp_fin":null,"ciudad":"Bilbao","zona_origen":"Centro","zona_destino":"Aeropuerto BIO","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-910","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"}
]"""

val ruta6 = s"$rutaJSON/viajes_bilbao_2024-11-01.json"
Files.write(Paths.get(ruta6), jsonBilbao.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta6")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_bilbao_2024-11-01.json


jsonBilbao: String = """[
  {"viaje_id":"VJ-BI-0001","timestamp_inicio":"2024-11-01T07:45:00","timestamp_fin":"2024-11-01T08:10:00","ciudad":"Bilbao","zona_origen":"Casco Viejo","zona_destino":"Guggenheim","distancia_km":2.8,"duracion_min":25,"tarifa_base":5.50,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-910","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:45:00","ciudad":"Bilbao","zona_origen":"Aeropuerto BIO","zona_destino":"Centro","distancia_km":11.5,"duracion_min":45,"tarifa_base":15.80,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","

In [8]:
// CELDA 3.7 — JSON multi-ciudad día 3
val jsonMultiCiudad =
"""[
  {"viaje_id":"VJ-M-0020","timestamp_inicio":"2024-11-03T08:00:00","timestamp_fin":"2024-11-03T08:30:00","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Retiro","distancia_km":3.8,"duracion_min":30,"tarifa_base":6.20,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0010","timestamp_inicio":"2024-11-03T09:15:00","timestamp_fin":"2024-11-03T09:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Sants","distancia_km":5.2,"duracion_min":35,"tarifa_base":8.80,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-610","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.1,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-S-0010","timestamp_inicio":"2024-11-03T10:00:00","timestamp_fin":"2024-11-03T10:25:00","ciudad":"Sevilla","zona_origen":"Nervión","zona_destino":"Centro","distancia_km":4.0,"duracion_min":25,"tarifa_base":6.50,"propina":0.50,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-7720","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-V-0010","timestamp_inicio":"2024-11-03T11:00:00","timestamp_fin":"2024-11-03T11:40:00","ciudad":"Valencia","zona_origen":"Centro","zona_destino":"Aeropuerto VLC","distancia_km":10.5,"duracion_min":40,"tarifa_base":14.80,"propina":3.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-803","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-BI-0010","timestamp_inicio":"2024-11-03T12:00:00","timestamp_fin":null,"ciudad":"Bilbao","zona_origen":"Guggenheim","zona_destino":"Casco Viejo","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-975","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-M-0021","timestamp_inicio":"2024-11-03T14:00:00","timestamp_fin":"2024-11-03T14:45:00","ciudad":"Madrid","zona_origen":"Chamartín","zona_destino":"Barajas","distancia_km":12.0,"duracion_min":45,"tarifa_base":16.00,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-332","pasajero_id":"PAX-5512","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0011","timestamp_inicio":"2024-11-03T16:30:00","timestamp_fin":"2024-11-03T17:00:00","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Barceloneta","distancia_km":4.7,"duracion_min":30,"tarifa_base":7.50,"propina":1.50,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-730","pasajero_id":"PAX-6631","tipo_vehiculo":"UberComfort","surge_multiplier":1.5,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-S-0011","timestamp_inicio":"2024-11-03T18:00:00","timestamp_fin":"2024-11-03T18:28:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Macarena","distancia_km":5.5,"duracion_min":28,"tarifa_base":8.20,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-561","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"}
]"""

val ruta7 = s"$rutaJSON/viajes_todas_ciudades_2024-11-03.json"
Files.write(Paths.get(ruta7), jsonMultiCiudad.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta7")

✅ Creado: C:/Curso-Scala/datos/rapidx/json_raw/viajes_todas_ciudades_2024-11-03.json


jsonMultiCiudad: String = """[
  {"viaje_id":"VJ-M-0020","timestamp_inicio":"2024-11-03T08:00:00","timestamp_fin":"2024-11-03T08:30:00","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Retiro","distancia_km":3.8,"duracion_min":30,"tarifa_base":6.20,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0010","timestamp_inicio":"2024-11-03T09:15:00","timestamp_fin":"2024-11-03T09:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Sants","distancia_km":5.2,"duracion_min":35,"tarifa_base":8.80,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-610","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.1,"_sys_version":"v3.3","_batch_id":"BA

In [9]:
// CELDA 3.8 — Verificación
val ficheros = new File(rutaJSON).listFiles().filter(_.getName.endsWith(".json")).sorted
println(s"Ficheros JSON en $rutaJSON:\n")
ficheros.foreach { f =>
  println(f"  ${f.getName}%-45s  ${f.length()} bytes")
}
println(s"\n✅ Total: ${ficheros.length} ficheros JSON listos")

Ficheros JSON en C:/Curso-Scala/datos/rapidx/json_raw:

  viajes_barcelona_2024-11-01.json               3514 bytes
  viajes_bilbao_2024-11-01.json                  2502 bytes
  viajes_madrid_2024-11-01.json                  4938 bytes
  viajes_madrid_2024-11-02.json                  2490 bytes
  viajes_sevilla_2024-11-01.json                 2479 bytes
  viajes_todas_ciudades_2024-11-03.json          3982 bytes

✅ Total: 6 ficheros JSON listos


ficheros: Array[File] = Array(
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_barcelona_2024-11-01.json,
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_bilbao_2024-11-01.json,
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_madrid_2024-11-01.json,
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_madrid_2024-11-02.json,
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_sevilla_2024-11-01.json,
  C:\Curso-Scala\datos\rapidx\json_raw\viajes_todas_ciudades_2024-11-03.json
)

In [10]:
// CELDA 4 — Leer todos los JSON
val dfRaw = spark.read
  .option("multiline", "false")   // un objeto JSON por línea — nuestro formato
  .json(s"$rutaJSON/*.json")

println("=== DataFrame RAW cargado desde JSON ===")
println(s"  Filas totales : ${dfRaw.count()}")
println(s"  Columnas      : ${dfRaw.columns.length}")
println()
println("Schema detectado automáticamente:")
dfRaw.printSchema()
println()
dfRaw.show(5, truncate = false)

=== DataFrame RAW cargado desde JSON ===
  Filas totales : 52
  Columnas      : 21

Schema detectado automáticamente:
root
 |-- _batch_id: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- _sys_version: string (nullable = true)
 |-- calificacion_conductor: long (nullable = true)
 |-- calificacion_pasajero: long (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- conductor_id: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: long (nullable = true)
 |-- estado: string (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- pasajero_id: string (nullable = true)
 |-- propina: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- tarifa_base: double (nullable = true)
 |-- timestamp_fin: string (nullable = true)
 |-- timestamp_inicio: string (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- viaje_id: string (nullable = true)
 |-- zona_destino: string (nullable = true

dfRaw: org.apache.spark.sql.package.DataFrame = [_batch_id: string, _corrupt_record: string ... 19 more fields]

In [11]:
// CELDA 5 — Enriquecimiento del DataFrame

val dfEnriquecido = dfRaw

  // ── 1. Parsear timestamps a tipo Timestamp ─────────────────────────────
  .withColumn("ts_inicio",  to_timestamp(col("timestamp_inicio"), "yyyy-MM-dd'T'HH:mm:ss"))
  .withColumn("ts_fin",     to_timestamp(col("timestamp_fin"),    "yyyy-MM-dd'T'HH:mm:ss"))

  // ── 2. Extraer fecha y hora de inicio ──────────────────────────────────
  .withColumn("fecha",      to_date(col("ts_inicio")))
  .withColumn("hora_inicio", hour(col("ts_inicio")))

  // ── 3. Franja horaria del viaje ────────────────────────────────────────
  //    Útil para BI: ¿cuándo hay más demanda?
  .withColumn("franja_horaria",
    when(col("hora_inicio").between(6, 9),   "Mañana pico")
    .when(col("hora_inicio").between(10, 13), "Mediodía")
    .when(col("hora_inicio").between(14, 17), "Tarde")
    .when(col("hora_inicio").between(18, 21), "Noche pico")
    .otherwise("Madrugada")
  )

  // ── 4. Importe total (tarifa + propina) ────────────────────────────────
  .withColumn("importe_total",
    round(col("tarifa_base") + col("propina"), 2)
  )

  // ── 5. Importe con surge aplicado ─────────────────────────────────────
  //    La tarifa_base ya lleva el surge, aquí lo dejamos explícito para BI
  .withColumn("importe_con_surge",
    round(col("tarifa_base") * col("surge_multiplier"), 2)
  )

  // ── 6. Indicador de viaje largo (distancia > 10 km) ───────────────────
  .withColumn("es_viaje_largo",
    col("distancia_km") > 10.0
  )

  // ── 7. Indicador de viaje con propina ─────────────────────────────────
  .withColumn("tiene_propina",
    col("propina") > 0.0
  )

  // ── 8. Segmento de calidad del conductor ──────────────────────────────
  //    Basado en calificacion_conductor (solo para viajes completados)
  .withColumn("segmento_conductor",
    when(col("estado") === "cancelado",          "Sin valorar")
    .when(col("calificacion_conductor") === 5,   "Excelente")
    .when(col("calificacion_conductor") >= 4,    "Bueno")
    .when(col("calificacion_conductor") >= 3,    "Aceptable")
    .otherwise("Bajo")
  )

  // ── 9. Indicador de surge activo ──────────────────────────────────────
  .withColumn("surge_activo",
    col("surge_multiplier") > 1.0
  )

  // ── 10. Precio por kilómetro ───────────────────────────────────────────
  //    Para viajes cancelados distancia_km = 0 → null para evitar división por cero
  .withColumn("precio_por_km",
    when(col("distancia_km") > 0.0,
      round(col("tarifa_base") / col("distancia_km"), 2)
    ).otherwise(null)
  )

println("=== DataFrame enriquecido ===")
println(s"  Columnas originales : ${dfRaw.columns.length}")
println(s"  Columnas tras enriquecimiento: ${dfEnriquecido.columns.length}")
println()
dfEnriquecido.select(
  "viaje_id", "ciudad", "fecha", "franja_horaria",
  "importe_total", "es_viaje_largo", "segmento_conductor", "surge_activo", "precio_por_km"
).show(8, truncate = false)

=== DataFrame enriquecido ===
  Columnas originales : 21
  Columnas tras enriquecimiento: 33

+---------+------+----------+--------------+-------------+--------------+------------------+------------+-------------+
|viaje_id |ciudad|fecha     |franja_horaria|importe_total|es_viaje_largo|segmento_conductor|surge_activo|precio_por_km|
+---------+------+----------+--------------+-------------+--------------+------------------+------------+-------------+
|NULL     |NULL  |NULL      |Madrugada     |NULL         |NULL          |Bajo              |NULL        |NULL         |
|VJ-M-0001|Madrid|2024-11-01|Mañana pico   |7.6          |false         |Excelente         |false       |1.33         |
|VJ-M-0002|Madrid|2024-11-01|Mañana pico   |5.1          |false         |Bueno             |true        |1.59         |
|VJ-M-0003|Madrid|2024-11-01|Mañana pico   |0.0          |false         |Sin valorar       |false       |NULL         |
|VJ-M-0004|Madrid|2024-11-01|Mañana pico   |11.2         |false   

dfEnriquecido: org.apache.spark.sql.package.DataFrame = [_batch_id: string, _corrupt_record: string ... 31 more fields]

In [12]:
// CELDA 6 — DataFrame final para BI (columnas seleccionadas)

val dfBI = dfEnriquecido.select(
  // Identificadores
  col("viaje_id"),
  col("fecha"),
  col("hora_inicio"),
  col("franja_horaria"),

  // Geografía
  col("ciudad"),
  col("zona_origen"),
  col("zona_destino"),

  // Operación
  col("tipo_vehiculo"),
  col("estado"),
  col("distancia_km"),
  col("duracion_min"),
  col("es_viaje_largo"),

  // Economía
  col("tarifa_base"),
  col("propina"),
  col("importe_total"),
  col("importe_con_surge"),
  col("surge_multiplier"),
  col("surge_activo"),
  col("tiene_propina"),
  col("precio_por_km"),
  col("metodo_pago"),

  // Calidad
  col("calificacion_pasajero"),
  col("calificacion_conductor"),
  col("segmento_conductor"),

  // Actores (anonimizados — solo IDs)
  col("conductor_id"),
  col("pasajero_id")
  // ❌ Excluidos intencionadamente:
  //   _batch_id       → campo interno del sistema operacional
  //   _sys_version    → campo interno del sistema operacional
  //   timestamp_inicio (String raw) → reemplazado por ts_inicio (Timestamp)
  //   timestamp_fin   (String raw) → reemplazado por ts_fin (Timestamp)
  //   ts_inicio, ts_fin → BI no los necesita, trabajan con fecha y hora_inicio
)

println("=== DataFrame BI final ===")
println(s"  Columnas seleccionadas: ${dfBI.columns.length}")
println(s"  Filas: ${dfBI.count()}")
println()
println("Columnas del DataFrame BI:")
dfBI.columns.zipWithIndex.foreach { case (col, i) =>
  println(f"  ${i + 1}%2d. $col")
}
println()
dfBI.show(5, truncate = true)

=== DataFrame BI final ===
  Columnas seleccionadas: 26
  Filas: 52

Columnas del DataFrame BI:
   1. viaje_id
   2. fecha
   3. hora_inicio
   4. franja_horaria
   5. ciudad
   6. zona_origen
   7. zona_destino
   8. tipo_vehiculo
   9. estado
  10. distancia_km
  11. duracion_min
  12. es_viaje_largo
  13. tarifa_base
  14. propina
  15. importe_total
  16. importe_con_surge
  17. surge_multiplier
  18. surge_activo
  19. tiene_propina
  20. precio_por_km
  21. metodo_pago
  22. calificacion_pasajero
  23. calificacion_conductor
  24. segmento_conductor
  25. conductor_id
  26. pasajero_id

+---------+----------+-----------+--------------+------+-----------+------------+-------------+----------+------------+------------+--------------+-----------+-------+-------------+-----------------+----------------+------------+-------------+-------------+-----------+---------------------+----------------------+------------------+------------+-----------+
| viaje_id|     fecha|hora_inicio|franja_

dfBI: org.apache.spark.sql.package.DataFrame = [viaje_id: string, fecha: date ... 24 more fields]

In [13]:
// CELDA 7 — Escritura Parquet particionado por ciudad
val rutaParquetBI = s"$rutaSalida/parquet_bi"

val inicio = System.nanoTime()

dfBI.write
  .mode("overwrite")
  .partitionBy("ciudad")
  .parquet(rutaParquetBI)

val tiempoMs = (System.nanoTime() - inicio) / 1_000_000L

println(s"✅ Parquet BI escrito en ${tiempoMs} ms")
println(s"   Ruta: $rutaParquetBI")
println()
println("Particiones generadas:")
new File(rutaParquetBI)
  .listFiles()
  .filter(_.isDirectory)
  .map(_.getName)
  .sorted
  .foreach(p => println(s"  $p/"))

✅ Parquet BI escrito en 1596 ms
   Ruta: C:/Curso-Scala/datos/rapidx/salida/parquet_bi

Particiones generadas:
  ciudad=Barcelona/
  ciudad=Bilbao/
  ciudad=Madrid/
  ciudad=Sevilla/
  ciudad=Valencia/
  ciudad=__HIVE_DEFAULT_PARTITION__/


rutaParquetBI: String = "C:/Curso-Scala/datos/rapidx/salida/parquet_bi"
inicio: Long = 15953204183900L
tiempoMs: Long = 1596L

In [14]:
// CELDA 8 — Registrar vista Spark SQL
val dfParquetBI = spark.read.parquet(rutaParquetBI)

dfParquetBI.createOrReplaceTempView("viajes_rapidx")

println("✅ Vista 'viajes_rapidx' registrada en Spark SQL")
println(s"   Filas en la vista: ${dfParquetBI.count()}")
println()
println("Schema de la vista:")
dfParquetBI.printSchema()

✅ Vista 'viajes_rapidx' registrada en Spark SQL
   Filas en la vista: 52

Schema de la vista:
root
 |-- viaje_id: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora_inicio: integer (nullable = true)
 |-- franja_horaria: string (nullable = true)
 |-- zona_origen: string (nullable = true)
 |-- zona_destino: string (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: long (nullable = true)
 |-- es_viaje_largo: boolean (nullable = true)
 |-- tarifa_base: double (nullable = true)
 |-- propina: double (nullable = true)
 |-- importe_total: double (nullable = true)
 |-- importe_con_surge: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- surge_activo: boolean (nullable = true)
 |-- tiene_propina: boolean (nullable = true)
 |-- precio_por_km: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- calificacion_pasajero: lo

dfParquetBI: org.apache.spark.sql.package.DataFrame = [viaje_id: string, fecha: date ... 24 more fields]

In [15]:
// CELDA 9.1 — Ingresos y volumen por ciudad
spark.sql("""
  SELECT
    ciudad,
    COUNT(*)                               AS total_viajes,
    SUM(CASE WHEN estado = 'completado'
              THEN 1 ELSE 0 END)           AS viajes_completados,
    SUM(CASE WHEN estado = 'cancelado'
              THEN 1 ELSE 0 END)           AS viajes_cancelados,
    ROUND(SUM(importe_total), 2)           AS ingresos_totales,
    ROUND(AVG(importe_total), 2)           AS ticket_medio,
    ROUND(AVG(distancia_km), 1)            AS distancia_media_km
  FROM viajes_rapidx
  GROUP BY ciudad
  ORDER BY ingresos_totales DESC
""").show(truncate = false)

+---------+------------+------------------+-----------------+----------------+------------+------------------+
|ciudad   |total_viajes|viajes_completados|viajes_cancelados|ingresos_totales|ticket_medio|distancia_media_km|
+---------+------------+------------------+-----------------+----------------+------------+------------------+
|Madrid   |17          |15                |2                |169.0           |9.94        |5.9               |
|Barcelona|9           |8                 |1                |89.0            |9.89        |5.6               |
|Sevilla  |7           |6                 |1                |50.5            |7.21        |4.6               |
|Bilbao   |6           |4                 |2                |40.3            |6.72        |3.7               |
|Valencia |1           |1                 |0                |18.3            |18.3        |10.5              |
|NULL     |12          |0                 |0                |NULL            |NULL        |NULL              |
+

In [16]:
// CELDA 9.2 — Distribución de viajes por franja horaria
spark.sql("""
  SELECT
    franja_horaria,
    COUNT(*)                               AS total_viajes,
    ROUND(SUM(importe_total), 2)           AS ingresos,
    ROUND(AVG(surge_multiplier), 2)        AS surge_medio,
    SUM(CASE WHEN surge_activo = true
              THEN 1 ELSE 0 END)           AS viajes_con_surge
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY franja_horaria
  ORDER BY total_viajes DESC
""").show(truncate = false)

+--------------+------------+--------+-----------+----------------+
|franja_horaria|total_viajes|ingresos|surge_medio|viajes_con_surge|
+--------------+------------+--------+-----------+----------------+
|Mañana pico   |13          |129.3   |1.06       |3               |
|Mediodía      |9           |110.3   |1.03       |1               |
|Tarde         |7           |79.4    |1.31       |4               |
|Noche pico    |4           |38.8    |1.38       |2               |
|Madrugada     |1           |9.3     |1.8        |1               |
+--------------+------------+--------+-----------+----------------+



In [17]:
// CELDA 9.3 — Análisis por tipo de vehículo
spark.sql("""
  SELECT
    tipo_vehiculo,
    COUNT(*)                               AS total_viajes,
    ROUND(AVG(distancia_km), 1)            AS distancia_media_km,
    ROUND(AVG(importe_total), 2)           AS ingreso_medio,
    ROUND(AVG(precio_por_km), 2)           AS precio_medio_km,
    ROUND(AVG(calificacion_pasajero), 2)   AS nota_media_pasajero
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY tipo_vehiculo
  ORDER BY ingreso_medio DESC
""").show(truncate = false)

+-------------+------------+------------------+-------------+---------------+-------------------+
|tipo_vehiculo|total_viajes|distancia_media_km|ingreso_medio|precio_medio_km|nota_media_pasajero|
+-------------+------------+------------------+-------------+---------------+-------------------+
|UberXL       |7           |12.2              |19.73        |1.34           |4.71               |
|UberComfort  |7           |6.4               |11.39        |1.54           |4.71               |
|UberX        |20          |4.3               |7.47         |1.66           |4.15               |
+-------------+------------+------------------+-------------+---------------+-------------------+



In [18]:
// CELDA 9.4 — Ranking de conductores
spark.sql("""
  SELECT
    conductor_id,
    COUNT(*)                               AS viajes_completados,
    ROUND(SUM(importe_total), 2)           AS ingresos_generados,
    ROUND(AVG(calificacion_pasajero), 2)   AS nota_media,
    segmento_conductor
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY conductor_id, segmento_conductor
  ORDER BY ingresos_generados DESC
  LIMIT 5
""").show(truncate = false)

+------------+------------------+------------------+----------+------------------+
|conductor_id|viajes_completados|ingresos_generados|nota_media|segmento_conductor|
+------------+------------------+------------------+----------+------------------+
|DRV-441     |4                 |41.4              |4.75      |Excelente         |
|DRV-119     |2                 |37.0              |5.0       |Excelente         |
|DRV-207     |3                 |29.4              |4.33      |Bueno             |
|DRV-975     |2                 |27.9              |4.5       |Excelente         |
|DRV-850     |1                 |27.0              |5.0       |Excelente         |
+------------+------------------+------------------+----------+------------------+



In [19]:
// CELDA 9.5 — Métodos de pago
spark.sql("""
  SELECT
    metodo_pago,
    COUNT(*)                               AS total_viajes,
    SUM(CASE WHEN tiene_propina = true
              THEN 1 ELSE 0 END)           AS viajes_con_propina,
    ROUND(
      100.0 * SUM(CASE WHEN tiene_propina = true THEN 1 ELSE 0 END)
            / COUNT(*), 1
    )                                       AS pct_con_propina,
    ROUND(AVG(CASE WHEN tiene_propina = true
                    THEN propina END), 2)  AS propina_media
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY metodo_pago
  ORDER BY total_viajes DESC
""").show(truncate = false)

+-----------+------------+------------------+---------------+-------------+
|metodo_pago|total_viajes|viajes_con_propina|pct_con_propina|propina_media|
+-----------+------------+------------------+---------------+-------------+
|tarjeta    |19          |17                |89.5           |2.25         |
|efectivo   |9           |3                 |33.3           |1.0          |
|app_wallet |6           |4                 |66.7           |1.25         |
+-----------+------------+------------------+---------------+-------------+



In [20]:
// CELDA 9.6 — Impacto del surge por ciudad
spark.sql("""
  SELECT
    ciudad,
    SUM(CASE WHEN surge_activo = true  THEN 1 ELSE 0 END)  AS viajes_con_surge,
    SUM(CASE WHEN surge_activo = false THEN 1 ELSE 0 END)  AS viajes_sin_surge,
    ROUND(AVG(CASE WHEN surge_activo = true
                    THEN surge_multiplier END), 2)          AS surge_medio_activo,
    ROUND(SUM(CASE WHEN surge_activo = true
                    THEN importe_con_surge ELSE 0 END), 2)  AS ingresos_extra_surge
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY ciudad
  ORDER BY ingresos_extra_surge DESC
""").show(truncate = false)

+---------+----------------+----------------+------------------+--------------------+
|ciudad   |viajes_con_surge|viajes_sin_surge|surge_medio_activo|ingresos_extra_surge|
+---------+----------------+----------------+------------------+--------------------+
|Madrid   |6               |9               |1.62              |94.19               |
|Barcelona|3               |5               |1.4               |37.25               |
|Sevilla  |1               |5               |1.4               |10.92               |
|Bilbao   |1               |3               |1.3               |10.53               |
|Valencia |0               |1               |NULL              |0.0                 |
+---------+----------------+----------------+------------------+--------------------+



In [21]:
// CELDA 9.7 — Análisis de cancelaciones
spark.sql("""
  SELECT
    ciudad,
    zona_origen,
    COUNT(*)                                          AS total_intentos,
    SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END) AS cancelados,
    ROUND(
      100.0 * SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END)
            / COUNT(*), 1
    )                                                  AS tasa_cancelacion_pct
  FROM viajes_rapidx
  GROUP BY ciudad, zona_origen
  HAVING COUNT(*) >= 1
  ORDER BY tasa_cancelacion_pct DESC
  LIMIT 10
""").show(truncate = false)

+---------+---------------+--------------+----------+--------------------+
|ciudad   |zona_origen    |total_intentos|cancelados|tasa_cancelacion_pct|
+---------+---------------+--------------+----------+--------------------+
|Barcelona|Sagrada Familia|1             |1         |100.0               |
|Madrid   |Vallecas       |1             |1         |100.0               |
|Madrid   |Moncloa        |1             |1         |100.0               |
|Bilbao   |Centro         |1             |1         |100.0               |
|Bilbao   |Guggenheim     |2             |1         |50.0                |
|Sevilla  |Nervión        |2             |1         |50.0                |
|Madrid   |Barajas        |2             |0         |0.0                 |
|Madrid   |Hortaleza      |1             |0         |0.0                 |
|Madrid   |Retiro         |2             |0         |0.0                 |
|Barcelona|Eixample       |2             |0         |0.0                 |
+---------+--------------

📝 Parte 10 — Preguntas de reflexión
Responde en celdas Markdown dentro de tu notebook antes de entregar:

Sobre el pipeline:

¿Por qué leemos los JSON con spark.read.json("carpeta/*.json") en lugar de leerlos uno a uno?Porque permite paralelizar la carga de archivos entre diferentes cluster .
¿Qué columnas del JSON original decidiste excluir del DataFrame BI y por qué?Las columns de tiempo de inicio y tiempo de fin ya que no aportan valor a los análisis de negocio y solo consumen almacenanamiento.
¿Qué pasaría si usaras inferSchema en los CSV pero aquí con JSON Spark infiere el schema automáticamente sin opción especial? ¿Cuál es la diferencia técnica?el json tinene métodos implícitos ,permitiendo a Spark identificar tipos básicos en una sola pasada.
Sobre las funciones de Spark:

En el enriquecimiento usamos when / otherwise para franja_horaria y segmento_conductor. ¿Por qué es mejor que escribir una UDF para esos casos?Porque con when/otherwise se ejecuta directamente en el catalyst.Un UDF obliga a Spark a serializar todos los datos .
La columna precio_por_km tiene un when(...).otherwise(null). ¿Qué ocurriría si no protegieras la división por cero para los viajes cancelados?Puede generar error de exportación a bases de datos relacionales.
¿En qué se diferencia to_timestamp de to_date? ¿Por qué necesitamos ambas?to_date extrae solo año-mes-día y to_timestamp matiene la precisión de horas ,minutos y segundos(esencial para franjas horarias)
Sobre el Parquet particionado:

Particionamos por ciudad. ¿Qué consulta SQL del equipo de BI se beneficia más de esta partición y por qué?las consltyas de Where ciudad ="..." son lás mas benficiadas porque puede obtener la infomación solo de la ciudad que se necesite y ignorando el resto de los datos en el disco.
Si el equipo de BI también filtra frecuentemente por fecha, ¿añadirías una segunda clave de partición? ¿Cuál sería el riesgo de particionar por fecha en este dataset?si el riesgo sería que si no hubiera mucha demanda de viajes tendriamos miles de archivos pequeños y sobrecargaría el sistema de archivos y haria que el progama fuera con más lentitud.
Sobre Spark SQL:

En la Consulta BI-5 calculamos el porcentaje de viajes con propina usando 100.0 * ... / COUNT(*). ¿Por qué usamos 100.0 en lugar de 100?
¿Cómo actualizarías la vista viajes_rapidx si RapidX añade los ficheros JSON de diciembre sin borrar los datos de noviembre?Se utiliza el 100.0 par forzaar la división decimal . Para mantener los ficheros de noviembre y diciembre usaríamos el metodo appender o apuntar a la raíz de la carpeta que contiene ambos meses .